<a href="https://colab.research.google.com/github/2303a51899/RL/blob/main/RL_LAB_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
gamma = 0.9
theta = 1e-6

states = ['S0', 'S1', 'Wall', 'G']
non_terminal_states = ['S0', 'S1']

goal_reward = 100.0
step_reward = -1.0
wall_reward = -1.0

transitions = {
    'S0': {
        'Up': ('S0', wall_reward),
        'Down': ('S1', step_reward),
        'Left': ('S0', wall_reward),
        'Right': ('Wall', wall_reward),
    },
    'S1': {
        'Up': ('S0', step_reward),
        'Down': ('S1', wall_reward),
        'Left': ('S1', wall_reward),
        'Right': ('G', goal_reward),
    },
}

print('=== 1. MDP INSPECTION ===')
print(f'All States          : {states}')
print(f'Non-Terminal States : {non_terminal_states}')
print(
    f'Rewards             : Step={step_reward}, Wall={wall_reward},'
    f' Goal={goal_reward}'
)
print(f'Discount Factor     : {gamma}\n')

print('=== 2. VERIFY TRANSITIONS ===')
for s in non_terminal_states:
  assert (
      s in transitions and len(transitions[s]) > 0
  ), f'Missing transitions for {s}'
  for a, (next_s, r) in transitions[s].items():
    print(f'State: {s:<3} | Action: {a:<6} -> Next State: {next_s:<4} | Reward: {r}')
print('All state-action pairs verified.\n')

V = {s: 0.0 for s in states}
print('=== 3. INITIAL STATE VALUES ===')
print(V, '\n')

print('=== 4 & 5. VALUE ITERATION (SYNCHRONOUS UPDATES) ===')
iteration = 0
while True:
  iteration += 1
  delta = 0.0
  V_new = V.copy()

  for s in non_terminal_states:
    action_values = [
        r + gamma * V[next_s] for next_s, r in transitions[s].values()
    ]
    V_new[s] = max(action_values)
    delta = max(delta, abs(V_new[s] - V[s]))

  V = V_new
  print(
      f'Iter {iteration:02d} | V(S0) = {V["S0"]:.4f} | V(S1) = {V["S1"]:.4f} |'
      f' Delta = {delta:.6f}'
  )

  if delta < theta:
    break
print()

print('=== 6. OPTIMAL ACTION VALUES Q*(s, a) ===')
Q = {s: {} for s in non_terminal_states}
for s in non_terminal_states:
  for a, (next_s, r) in transitions[s].items():
    Q[s][a] = r + gamma * V[next_s]
    print(f'Q*({s}, {a:<5}) = {Q[s][a]:.4f}')
print()

print('=== 7. OPTIMAL POLICY ===')
optimal_policy = {}
for s in non_terminal_states:
  max_q = max(Q[s].values())
  best_actions = [
      a for a, q_val in Q[s].items() if abs(q_val - max_q) < 1e-9
  ]
  optimal_policy[s] = best_actions
  print(f'pi*({s}) = {best_actions}')
print()

print('=== 8. BELLMAN OPTIMALITY VERIFICATION ===')
for s in non_terminal_states:
  max_q = max(Q[s].values())
  is_equal = abs(V[s] - max_q) < 1e-9
  print(
      f'{s}: V*({s}) = {V[s]:.4f}, max_a Q*({s}, a) = {max_q:.4f} -> Verified:'
      f' {is_equal}'
  )

=== 1. MDP INSPECTION ===
All States          : ['S0', 'S1', 'Wall', 'G']
Non-Terminal States : ['S0', 'S1']
Rewards             : Step=-1.0, Wall=-1.0, Goal=100.0
Discount Factor     : 0.9

=== 2. VERIFY TRANSITIONS ===
State: S0  | Action: Up     -> Next State: S0   | Reward: -1.0
State: S0  | Action: Down   -> Next State: S1   | Reward: -1.0
State: S0  | Action: Left   -> Next State: S0   | Reward: -1.0
State: S0  | Action: Right  -> Next State: Wall | Reward: -1.0
State: S1  | Action: Up     -> Next State: S0   | Reward: -1.0
State: S1  | Action: Down   -> Next State: S1   | Reward: -1.0
State: S1  | Action: Left   -> Next State: S1   | Reward: -1.0
State: S1  | Action: Right  -> Next State: G    | Reward: 100.0
All state-action pairs verified.

=== 3. INITIAL STATE VALUES ===
{'S0': 0.0, 'S1': 0.0, 'Wall': 0.0, 'G': 0.0} 

=== 4 & 5. VALUE ITERATION (SYNCHRONOUS UPDATES) ===
Iter 01 | V(S0) = -1.0000 | V(S1) = 100.0000 | Delta = 100.000000
Iter 02 | V(S0) = 89.0000 | V(S1) = 100.0